# The MCP retrieval server

The tools the agent sees, called the way the agent calls them.

See the [README](README.md) for the why and the how.

In [ ]:
# Run from anywhere inside the repository: find its root so `marginalia` and the test helpers import,
# and read .env (the same settings the services use).
import os, sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'pyproject.toml').exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print(ROOT)

In [ ]:
from fastmcp import Client
from marginalia.mcp_server.server import build_server
from tests.conftest import book_retriever, synthetic_profile, synthetic_book
retriever = book_retriever('mcp-notebook')
chapters, _ = synthetic_book()
server = build_server(synthetic_profile(), lambda: retriever,
                      chapter_texts=lambda: {(c.part, c.chapter): c.text for c in chapters})
async with Client(server) as c:
    tools = await c.list_tools()
for t in tools:
    print(f'{t.name:26} {t.description.splitlines()[0]}')
print()
print(server.instructions)       # what the agent is told about this book

## Calling tools

In [ ]:
async with Client(server) as c:
    hits = (await c.call_tool('search_passages', {'query': 'Lazarus Gospel', 'limit': 2})).data
    around = (await c.call_tool('get_surrounding_text', {'passage_id': hits[0]['id'], 'before': 1, 'after': 0})).data
    toc = (await c.call_tool('list_chapters', {})).data
print([h['citation'] for h in hits])
print([p['citation'] for p in around])
print([(c['citation'], c['passages']) for c in toc])

## Mistakes come back as instructions
A wrong chapter or a chapter that never mentions the query is an error message the model can act on.

In [ ]:
async with Client(server) as c:
    for args in ({'query': 'Lazarus', 'part': 1, 'chapter': 9}, {'query': 'Lazarus', 'part': 1, 'chapter': 1}):
        res = await c.call_tool('search_passages', args, raise_on_error=False)
        print(args, '->', res.content[0].text)

## As its own process
The worker starts one server per book over stdio:

```bash
python -m marginalia.mcp_server --book crime
```